# SignBridge M3 — WLASL Landmark Shards (Optimized Full 12K Extraction)

This Colab preprocesses the full official WLASL archive (ASL, ~12,000 videos, 2,000 words) into the SignBridge M3 landmark contract.
It is heavily optimized with:
- **Hardware GPU Delegate**: Native MediaPipe GPU acceleration on NVIDIA T4.
- **Fast Video Seeking**: `cv2.CAP_PROP_POS_FRAMES` instant seek directly to start frame (eliminates decoding thousands of dead frames on long YouTube clips).
- **Fast Frame Downscaling**: Downscales high-resolution frames to 256px max dimension (MediaPipe normalized coordinates are invariant; cuts decode and texture copy time by ~3x).
- **Safe Resume**: Progress is written in shards (`SHARD_SIZE=64`). If disconnected, rerunning continues from the last completed video without losing progress.


In [ ]:
!pip -q install mediapipe==0.10.35 opencv-contrib-python==5.0.0.93 numpy==2.3.5 requests


In [ ]:
from pathlib import Path
import os, json, time, zipfile, shutil, urllib.request
import numpy as np

BENCHMARK = '2000'
SUBSET = 'all'
FIRST_INSTANCE_ONLY = False
FAST_SEEK = True
RESIZE_MAX_DIM = 256
RESUME = True
ZIP_PATH = None
DRIVE_ZIP_PATH = None
OUTPUT_DIR = Path('/content/signbridge_wlasl_m3')
OVERWRITE = False
MAX_FRAMES = 60
FEATURE_SHAPE = (110, 3)
HAND_SLOTS = ['left', 'right']
FACE_POINTS = 478
COMPACT_FACE_POINTS = 68
SHARD_SIZE = 64
LANDMARK_VERSION = 'compact-v1'
FACE_LANDMARK_INDICES = {'RIGHT_EYE': [33, 7, 163, 144, 145, 153, 154, 155, 133, 173, 157, 158, 159, 160, 161, 246], 'LEFT_EYE': [263, 249, 390, 373, 374, 380, 381, 382, 362, 398, 384, 385, 386, 387, 388, 466], 'RIGHT_EYEBROW': [70, 63, 105, 66, 107], 'LEFT_EYEBROW': [336, 296, 334, 293, 300], 'LIPS_OUTER': [61, 146, 91, 181, 84, 17, 314, 405, 321, 375], 'LIPS_INNER': [78, 95, 88, 178, 87, 14, 317, 402, 318, 324], 'NOSE': [1, 2, 4, 5, 6, 168]}
COMPACT_FACE_INDICES = [i for group in FACE_LANDMARK_INDICES.values() for i in group]
assert len(COMPACT_FACE_INDICES) == COMPACT_FACE_POINTS
HAND_URL = 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task'
FACE_URL = 'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task'
MODEL_MIN_BYTES = 100_000


In [ ]:
# Drive-only archive input. This cell never opens the Colab upload picker.
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ZIP_PATH = '/content/drive/MyDrive/archive (1).zip'
ZIP_PATH = Path(DRIVE_ZIP_PATH)
if not ZIP_PATH.is_file():
    print('ZIP not found at:', ZIP_PATH)
    print('ZIP files found in Drive:')
    for candidate in Path('/content/drive/MyDrive').rglob('*.zip'):
        print(candidate)
    raise FileNotFoundError(f'Update DRIVE_ZIP_PATH; file not found: {ZIP_PATH}')
print('Using ZIP directly from Drive:', ZIP_PATH)
print('Size GB:', round(ZIP_PATH.stat().st_size / (1024 ** 3), 2))


In [ ]:
def safe_extract(zip_path, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    if (destination / 'videos').is_dir() and (destination / 'WLASL_v0.3.json').is_file():
        print('Archive already extracted at destination, skipping unzip.')
        return
    import subprocess
    print('Extracting ZIP archive with fast system unzip...', flush=True)
    ret = subprocess.run(['unzip', '-q', '-o', str(zip_path), '-d', str(destination)])
    if ret.returncode != 0:
        print('Falling back to Python zipfile...', flush=True)
        with zipfile.ZipFile(zip_path) as z:
            for info in z.infolist():
                target = (destination / info.filename).resolve()
                if target != destination and destination not in target.parents:
                    raise ValueError(f'unsafe ZIP member: {info.filename!r}')
                if info.is_dir(): target.mkdir(parents=True, exist_ok=True)
                else:
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with z.open(info) as src, open(target, 'wb') as dst: shutil.copyfileobj(src, dst)
    print('Extraction complete.', flush=True)

extract_root = Path('/content/wlasl_extract')
safe_extract(ZIP_PATH, extract_root)

def find_root(base):
    for p in [base] + [x for x in base.rglob('*') if x.is_dir()]:
        if (p / 'videos').is_dir() and (p / 'wlasl_class_list.txt').is_file() and (p / 'WLASL_v0.3.json').is_file():
            return p
    raise FileNotFoundError('Could not find archive root with videos/, class list, and WLASL_v0.3.json')

ARCHIVE_ROOT = find_root(extract_root)
print('Archive root:', ARCHIVE_ROOT)


In [ ]:
def download_asset(url,path):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists() and path.stat().st_size>=MODEL_MIN_BYTES: return
    tmp=path.with_suffix(path.suffix+'.tmp'); req=urllib.request.Request(url,headers={'User-Agent':'SignBridge-Colab/1'})
    with urllib.request.urlopen(req,timeout=120) as r:
        if r.headers.get('Content-Length') and int(r.headers['Content-Length'])<MODEL_MIN_BYTES: raise RuntimeError(f'asset too small: {url}')
        with open(tmp,'wb') as f: shutil.copyfileobj(r,f)
    if tmp.stat().st_size<MODEL_MIN_BYTES: tmp.unlink(missing_ok=True); raise RuntimeError(f'asset size check failed: {url}')
    os.replace(tmp,path); print(path.name,path.stat().st_size,'bytes',flush=True)
MODEL_DIR=Path('/content/signbridge_models'); HAND_MODEL=MODEL_DIR/'hand_landmarker.task'; FACE_MODEL=MODEL_DIR/'face_landmarker.task'
download_asset(HAND_URL,HAND_MODEL); download_asset(FACE_URL,FACE_MODEL)


In [ ]:
def class_list(path):
    out = {}
    for n, line in enumerate(Path(path).read_text(encoding='utf-8').splitlines(), 1):
        if line.strip() and not line.lstrip().startswith('#'):
            parts = line.split(None, 1)
            if len(parts) != 2: raise ValueError(f'malformed class list line {n}')
            out[int(parts[0])] = parts[1].strip()
    if not out: raise ValueError('empty class list')
    return out

def master_metadata(root):
    data = json.loads((root / 'WLASL_v0.3.json').read_text(encoding='utf-8'))
    out = {}
    for entry in data if isinstance(data, list) else data.values():
        for inst in entry.get('instances', []):
            if 'video_id' in inst: out[str(inst['video_id'])] = {**inst, 'gloss': entry.get('gloss')}
    return out

classes = class_list(ARCHIVE_ROOT / 'wlasl_class_list.txt')
master = master_metadata(ARCHIVE_ROOT)
bench_path = Path(BENCHMARK) if Path(BENCHMARK).is_file() else ARCHIVE_ROOT / f'nslt_{BENCHMARK}.json'
if not bench_path.is_file(): raise FileNotFoundError(bench_path)

records = []
bench = json.loads(bench_path.read_text(encoding='utf-8'))
for vid, item in bench.items():
    action = item.get('action', [])
    if not action: continue
    original = int(action[0])
    meta = master.get(str(vid), master.get(str(vid).zfill(5), {}))
    vp = ARCHIVE_ROOT / 'videos' / f'{vid}.mp4'
    if not vp.is_file() and str(vid).isdigit(): vp = ARCHIVE_ROOT / 'videos' / f'{int(vid):05d}.mp4'
    if vp.is_file():
        records.append({
            'video_id': str(vid),
            'video_path': vp,
            'subset': item.get('subset', meta.get('split', '')),
            'original_class_id': original,
            'gloss': classes.get(original, meta.get('gloss', str(original))),
            'signer_id': meta.get('signer_id'),
            'instance_id': meta.get('instance_id', 999),
            'frame_start': action[1] if len(action) > 1 else meta.get('frame_start'),
            'frame_end': action[2] if len(action) > 2 else meta.get('frame_end')
        })

if SUBSET != 'all':
    records = [r for r in records if r['subset'] == SUBSET]

remap = {old: i for i, old in enumerate(sorted({r['original_class_id'] for r in records}))}
for r in records: r['class_id'] = remap[r['original_class_id']]

print(json.dumps({
    'benchmark': str(BENCHMARK),
    'subset': SUBSET,
    'first_instance_only': FIRST_INSTANCE_ONLY,
    'usable_records': len(records),
    'classes_complete_benchmark': len(remap)
}, indent=2), flush=True)


## Extraction and sharded writer
MediaPipe Tasks IMAGE mode emits `(110, 3)` frames: left/right hand slots followed by 68 selected compact-v1 face points from the validated 478-point source. Benchmark ranges are 1-based inclusive and applied before the fixed 60-frame pad/truncate.
compact-v1 selects 68 face points from the validated 478-point source; missing faces are zero-filled. NPZ X=(batch,60,110,3), y=int64. Progress reports every 500 frames with ETA.


In [ ]:
import cv2, mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

def frame_points(hr, fr):
    out = np.zeros(FEATURE_SHAPE, np.float32)
    for i, hand in enumerate(getattr(hr, 'hand_landmarks', []) or []):
        labels = getattr(hr, 'handedness', []) or []
        label = getattr(labels[i][0], 'category_name', '').lower() if i < len(labels) and labels[i] else ''
        slot = 0 if label == 'left' else 1 if label == 'right' else i
        if slot < 2: out[slot * 21:(slot + 1) * 21] = [[p.x, p.y, p.z] for p in hand]
    faces = getattr(fr, 'face_landmarks', []) or []
    if faces and len(faces[0]) == FACE_POINTS:
        out[42:] = [[faces[0][k].x, faces[0][k].y, faces[0][k].z] for k in COMPACT_FACE_INDICES]
    return out

class Progress:
    def __init__(self, total, interval=500):
        self.total = total
        self.interval = interval
        self.next = interval
        self.processed = 0
        self.started = time.time()
    def update(self, n=1):
        self.processed += n
        while self.processed >= self.next:
            elapsed = time.time() - self.started
            fps = self.processed / max(elapsed, 1e-9)
            remain = max(self.total - self.processed, 0)
            eta_m = (remain / max(fps, 1e-9)) / 60.0
            print(f'progress: elapsed {elapsed/60:.1f}m | {fps:.1f} fps | {self.processed}/{self.total} frames | ETA {eta_m:.1f} min', flush=True)
            self.next += self.interval

def estimate_selected(rec):
    if rec['frame_start'] is not None and rec['frame_end'] is not None:
        return min(MAX_FRAMES, max(0, int(rec['frame_end']) - int(rec['frame_start']) + 1))
    cap = cv2.VideoCapture(str(rec['video_path']))
    count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    cap.release()
    start = max(1, int(rec['frame_start'] or 1))
    return min(MAX_FRAMES, max(0, count - start + 1)) if count else MAX_FRAMES

def extract(video, hand, face, start=None, end=None, progress=None, max_dim=256):
    cap = cv2.VideoCapture(str(video))
    if not cap.isOpened(): raise ValueError('video could not be opened')
    seq = []
    total_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    start = max(1, int(start or 1))
    end = int(end) if end is not None and int(end) > 0 else (total_count if total_count > 0 else None)
    
    # Fast seek straight to start frame
    if FAST_SEEK and start > 1 and total_count > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start - 1)
        number = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
    else:
        number = 0
        
    try:
        while len(seq) < MAX_FRAMES:
            ok, bgr = cap.read()
            if not ok: break
            number += 1
            if number < start: continue
            if end is not None and number > end: break
            
            # Downscale large frames to max_dim (256px) for speedup (normalized coords are invariant)
            if max_dim:
                h, w = bgr.shape[:2]
                if max(h, w) > max_dim:
                    scale = max_dim / float(max(h, w))
                    bgr = cv2.resize(bgr, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
                    
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            seq.append(frame_points(hand.detect(image), face.detect(image)))
            if progress: progress.update()
    finally:
        cap.release()
    if not seq: raise ValueError('selected range yielded no frames')
    arr = np.zeros((MAX_FRAMES, *FEATURE_SHAPE), np.float32)
    arr[:len(seq)] = np.asarray(seq, np.float32)
    return arr

def atomic_npz(path, X, y):
    tmp = Path(path).with_suffix('.npz.tmp')
    with open(tmp, 'wb') as f: np.savez(f, X=np.asarray(X, np.float32), y=np.asarray(y, np.int64))
    os.replace(tmp, path)


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
index_path = OUTPUT_DIR / 'dataset.index.json'
manifest_path = OUTPUT_DIR / 'dataset.manifest.jsonl'
failures_path = OUTPUT_DIR / 'failures.jsonl'

processed_vids = set()
if RESUME and not OVERWRITE:
    if manifest_path.exists():
        with open(manifest_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    try: processed_vids.add(json.loads(line)['video_id'])
                    except Exception: pass
    if failures_path.exists():
        with open(failures_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    try: processed_vids.add(json.loads(line)['video_id'])
                    except Exception: pass
    if processed_vids:
        print(f'Resuming: found {len(processed_vids)} already processed videos. Skipping them.', flush=True)
elif OVERWRITE:
    for p in list(OUTPUT_DIR.glob('shard-*.npz')) + [p for p in (index_path, manifest_path, failures_path) if p.exists()]:
        p.unlink()

active_records = [r for r in records if r['video_id'] not in processed_vids]
print(f'Active videos to process: {len(active_records)} / {len(records)} total', flush=True)

total_estimated_frames = sum(estimate_selected(r) for r in active_records)
progress = Progress(total_estimated_frames)

delegate = python.BaseOptions.Delegate.CPU
try:
    import subprocess
    if subprocess.run(['nvidia-smi'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0:
        delegate = python.BaseOptions.Delegate.GPU
        print('GPU acceleration detected: using MediaPipe GPU Delegate (NVIDIA T4)!', flush=True)
except Exception:
    print('Running on CPU Delegate.', flush=True)

hand_opts = vision.HandLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path=str(HAND_MODEL), delegate=delegate),
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2
)
face_opts = vision.FaceLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path=str(FACE_MODEL), delegate=delegate),
    running_mode=vision.RunningMode.IMAGE,
    num_faces=1
)

existing_shards = sorted([p.name for p in OUTPUT_DIR.glob('shard-*.npz')])
shards = list(existing_shards)
labels = {r['class_id']: r['gloss'] for r in records}
counts = {}
failures = 0
batchX, batchY, batchM = [], [], []

def flush_batch(mf):
    global batchX, batchY, batchM
    if not batchX: return
    name = f'shard-{len(shards):05d}.npz'
    atomic_npz(OUTPUT_DIR / name, batchX, batchY)
    shards.append(name)
    for row, m in enumerate(batchM):
        mf.write(json.dumps({k: m[k] for k in ('video_id', 'subset', 'class_id', 'original_class_id', 'gloss', 'signer_id', 'frame_start', 'frame_end')} | {'shard': name, 'row_in_shard': row}) + '\n')
    mf.flush()
    batchX, batchY, batchM = [], [], []

mode_manifest = 'a' if (RESUME and manifest_path.exists()) else 'w'
mode_failures = 'a' if (RESUME and failures_path.exists()) else 'w'

with vision.HandLandmarker.create_from_options(hand_opts) as hand, vision.FaceLandmarker.create_from_options(face_opts) as face, open(manifest_path, mode_manifest, encoding='utf-8') as mf, open(failures_path, mode_failures, encoding='utf-8') as ff:
    for rec in active_records:
        try:
            batchX.append(extract(rec['video_path'], hand, face, rec['frame_start'], rec['frame_end'], progress, max_dim=RESIZE_MAX_DIM))
            batchY.append(rec['class_id'])
            batchM.append(rec)
            counts[rec['class_id']] = counts.get(rec['class_id'], 0) + 1
            if len(batchX) >= SHARD_SIZE: flush_batch(mf)
        except Exception as e:
            failures += 1
            ff.write(json.dumps({'video_id': rec['video_id'], 'error': f'{type(e).__name__}: {e}'}) + '\n')
            ff.flush()
    flush_batch(mf)

total_samples = 0
if manifest_path.exists():
    with open(manifest_path, 'r', encoding='utf-8') as f:
        total_samples = sum(1 for line in f if line.strip())

index = {
    'format': 'signbridge-m3-sharded',
    'version': 1,
    'MAX_FRAMES': MAX_FRAMES,
    'feature_shape': list(FEATURE_SHAPE),
    'hand_slots': HAND_SLOTS,
    'face_points': COMPACT_FACE_POINTS,
    'benchmark': str(BENCHMARK),
    'subset': SUBSET,
    'first_instance_only': FIRST_INSTANCE_ONLY,
    'labels': {str(k): v for k, v in sorted(labels.items())},
    'per_label_counts': {str(k): v for k, v in sorted(counts.items())},
    'sample_count': total_samples,
    'failed_count': failures,
    'shards': shards,
    'landmark_version': LANDMARK_VERSION,
    'face_landmark_indices': FACE_LANDMARK_INDICES,
    'face_landmark_order': COMPACT_FACE_INDICES,
    'face_points_source': FACE_POINTS,
    'face_points_selected': COMPACT_FACE_POINTS
}
index_path.write_text(json.dumps(index, indent=2) + '\n', encoding='utf-8')
print(json.dumps({
    'landmark_version': LANDMARK_VERSION,
    'X_shape': ['batch', MAX_FRAMES, *FEATURE_SHAPE],
    'y_dtype': 'int64',
    'samples_written': index['sample_count'],
    'shards': len(shards),
    'failures': failures
}, indent=2), flush=True)


In [ ]:
def iter_samples(output_dir=OUTPUT_DIR):
    idx=json.loads((Path(output_dir)/'dataset.index.json').read_text())
    for name in idx['shards']:
        with np.load(Path(output_dir)/name) as z:
            for X,y in zip(z['X'],z['y']): yield X,y
